In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import timm

MODEL_DIR = Path(r'D:\Tree-Structured Parzen Estimator\main\[6] model')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CKPT_FILES = {
    'densenet121' : MODEL_DIR / 'densenet121_best.pth',
    'inception_v3': MODEL_DIR / 'inception_v3_best.pth',
    'xception'    : MODEL_DIR / 'xception_best.pth',
    'vit_scratch' : MODEL_DIR / 'vit_best.pth',
}

BEST_WEIGHTS = {
    'densenet121' : 0.4706,
    'inception_v3': 0.0893,
    'xception'    : 0.2850,
    'vit_scratch' : 0.1551,
}
MODEL_KEYS   = ['densenet121', 'inception_v3', 'xception', 'vit_scratch']
MODEL_LABELS = {
    'densenet121' : 'DenseNet-121',
    'inception_v3': 'InceptionV3',
    'xception'    : 'Xception',
    'vit_scratch' : 'ViT-Pretrained',
}
IMG_SIZE = {
    'densenet121' : 224,
    'inception_v3': 299,
    'xception'    : 299,
    'vit_scratch' : 224,
}

NUM_CLASSES = 2
VIT_MODEL_NAME = 'vit_small_patch16_224'


class CNNWithHead(nn.Module):
    def __init__(self, backbone, n_features, dropout, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(n_features, 512), nn.BatchNorm1d(512), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def build_cnn(arch_name, dropout):
    backbone   = timm.create_model(arch_name, pretrained=False, drop_rate=dropout, num_classes=0)
    n_features = backbone.num_features
    return CNNWithHead(backbone, n_features, dropout, NUM_CLASSES)


def build_vit_pretrained(dropout):
    model = timm.create_model(
        VIT_MODEL_NAME, pretrained=False, drop_rate=dropout,
        attn_drop_rate=dropout * 0.5, drop_path_rate=0.1, num_classes=NUM_CLASSES)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Linear(in_features, 256), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(256, NUM_CLASSES))
    return model


def load_submodel(key: str) -> nn.Module:
    path = CKPT_FILES[key]
    if not path.exists():
        raise FileNotFoundError(f'Checkpoint tidak ditemukan: {path}')

    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    bp   = ckpt['best_params']

    if key == 'vit_scratch':
        model = build_vit_pretrained(bp['dropout'])
    else:
        model = build_cnn(key, bp['dropout'])

    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f'  [OK] {MODEL_LABELS[key]:<16} loaded  (best_epoch={ckpt["best_epoch"]}, '
          f'val_acc={ckpt.get("best_val_acc", max(ckpt["history"]["val_acc"])):.4f})')
    return model


print('Loading 4 sub-model...')
submodels = {key: load_submodel(key) for key in MODEL_KEYS}


class EnsembleModel(nn.Module):
    """
    Weighted-average ensemble dari 4 model dengan img_size berbeda-beda.
    forward() menerima dict input per arsitektur karena DenseNet/ViT pakai 224x224
    sedangkan InceptionV3/Xception pakai 299x299 — TIDAK bisa 1 tensor untuk semua.

    Contoh pakai:
        inputs = {
            'densenet121' : batch_224,
            'inception_v3': batch_299,
            'xception'    : batch_299,
            'vit_scratch' : batch_224,
        }
        probs = ensemble(inputs)   # -> P(malignant), shape (batch,)
    """
    def __init__(self, models: dict, weights: dict, img_size: dict):
        super().__init__()
        self.models   = nn.ModuleDict(models)
        self.weights  = weights
        self.img_size = img_size
        self.keys     = list(models.keys())

    def forward(self, inputs: dict):
        probs = None
        for key in self.keys:
            logits = self.models[key](inputs[key])
            p = torch.softmax(logits, dim=1)[:, 1]
            probs = p * self.weights[key] if probs is None else probs + p * self.weights[key]
        return probs


ensemble = EnsembleModel(submodels, BEST_WEIGHTS, IMG_SIZE).to(DEVICE)
ensemble.eval()

save_path = MODEL_DIR / 'ensemble_best.pth'
torch.save({
    'state_dict'  : ensemble.state_dict(),
    'weights'     : BEST_WEIGHTS,
    'model_keys'  : MODEL_KEYS,
    'model_labels': MODEL_LABELS,
    'img_size'    : IMG_SIZE,
    'formula'     : '0.08*AUC + 0.05*Recall + 0.18*F1 + 0.69*Acc (Trial-460, selection score 0.990766)',
    'source_trial': 'Trial-460',
    'num_classes' : NUM_CLASSES,
    'vit_model_name': VIT_MODEL_NAME,
}, save_path)

print(f'\nEnsembleModel tersimpan -> {save_path}')
print(f'Ukuran file: {save_path.stat().st_size / 1e6:.2f} MB')
print(f'Bobot yang dipakai: {BEST_WEIGHTS}')

Loading 4 sub-model...
  [OK] DenseNet-121     loaded  (best_epoch=31, val_acc=0.9214)
  [OK] InceptionV3      loaded  (best_epoch=32, val_acc=0.9158)


c:\Python312\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


  [OK] Xception         loaded  (best_epoch=32, val_acc=0.9237)
  [OK] ViT-Pretrained   loaded  (best_epoch=25, val_acc=0.9360)

EnsembleModel tersimpan -> D:\Tree-Structured Parzen Estimator\main\[6] model\ensemble_best.pth
Ukuran file: 297.21 MB
Bobot yang dipakai: {'densenet121': 0.4706, 'inception_v3': 0.0893, 'xception': 0.285, 'vit_scratch': 0.1551}
